In [ ]:
import open3d as o3d
import numpy as np

# ================= 0. 强制重新读取和降采样 (请改成你的真实路径!) =================
print("[0] 重新读取点云...")
pcd = o3d.io.read_point_cloud("day9_curv.pcd")  # <--- 改成你的路径！
pcd = pcd.voxel_down_sample(voxel_size=0.03)

# 检查点云是否为空
if not pcd.has_points():
    raise ValueError("❌ 错误：点云为空！请检查你的文件路径是否正确！")
print(f"[0] 点云读取成功，点数: {len(np.asarray(pcd.points))}")

# ================= 1. 估计法线 (带参数和自检) =================
print("[1] 开始估计法线...")
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.15, max_nn=30)
)

# 核心自检：检查法线是否真的生成了
if not pcd.has_normals():
    raise RuntimeError("❌ 错误：estimate_normals 执行了但法线依然为空！请检查点云质量或搜索半径！")
else:
    norms = np.asarray(pcd.normals)
    print(f"[1] 法线估计成功！法线数量: {len(norms)}")
    print(f"[1] 法线预览(前3个): \n{norms[:3]}")

# ================= 2. 一致定向 =================
print("[2] 开始一致定向...")
# 确保这里使用的是 o3d.geometry 的常规对象，不要用 o3d.t.geometry
pcd.orient_normals_consistent_tangent_plane(k=15)
print("[2] 定向完成！")


def normals_lineset(pcd, normal_scale=0.01, color=[1, 0, 0]):
    line_set = o3d.geometry.LineSet()
    start = np.asarray(pcd.points)
    end = start + (np.asarray(pcd.normals) * normal_scale) # 法线通常是单位向量，需要缩放
    points = np.concatenate((start, end))
    line_set.points = o3d.utility.Vector3dVector(points)
    size = len(start)
    line_set.lines = o3d.utility.Vector2iVector(np.asarray([[i, i+size] for i in range(0, size)]))
    line_set.paint_uniform_color(color)
    return line_set

# 把点云和法线线段一起画出来
o3d.visualization.draw([pcd, normals_lineset(pcd)])
# ================= 3. 可视化 =================
print("[3] 打开可视化窗口...")
o3d.visualization.draw([pcd])

[0] 重新读取点云...
[0] 点云读取成功，点数: 7516
[1] 开始估计法线...
[1] 法线估计成功！法线数量: 7516
[1] 法线预览(前3个): 
[[ 0.01891864  0.3013155  -0.9533368 ]
 [ 0.14285528 -0.2403399  -0.96011932]
 [ 0.14702025  0.14702025 -0.97814625]]
[2] 开始一致定向...
[2] 定向完成！
[Open3D INFO] Window window_0 created.
[Open3D INFO] ICE servers: ["stun:stun.l.google.com:19302", "turn:user:password@34.69.27.100:3478", "turn:user:password@34.69.27.100:3478?transport=tcp"]
[Open3D INFO] Set WEBRTC_STUN_SERVER environment variable add a customized WebRTC STUN server.
[Open3D INFO] WebRTC Jupyter handshake mode enabled.
